# recheck_comlumn_preapare — ตรวจ layer + แตก process parameter

recheck ผลของ `column_prepare.ipynb` อ่านจาก `processed/prepared/` (ไม่แก้ต้นทาง)

4 เรื่อง
1. layer หลัก vs layer เคลือบผิว + จำนวนชั้น
2. คอลัมน์ใหม่ระดับ device: เคลือบ / ไม่เคลือบ (layer 3,5,7 = ชั้นเคลือบ)
3. layer 2,4 ปิดด้วย Annealing จริงไหม
4. แตก `process.parameter` + `unit` (เช่น `155, 600, 600` / `W, mtorr, sec`) เป็นคอลัมน์แยก

## Setup

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "prepared").exists() else Path.cwd().parent
SRC = ROOT / "processed" / "prepared"
tp = {p.stem: pd.read_csv(p) for p in sorted(SRC.glob("*.csv"))}
lay, pr = tp["layers"], tp["process"]
print("โหลดจาก:", SRC)
display(pd.DataFrame({n: d.shape for n, d in tp.items()}, index=["rows", "cols"]).T)

## 1. layer หลัก vs layer เคลือบผิว

ตำแหน่ง layer_id เป็นระบบ: **เลขคู่ = ชั้นหลัก, เลขคี่กลาง = ชั้นเคลือบ** (แทรกระหว่างชั้นหลัก)

| layer_id | บทบาท | ชนิด |
|---|---|---|
| 1 | Anode | หลัก |
| 2 | HTL | หลัก |
| **3** | **HTL treatment** | **เคลือบ** |
| 4 | PSK | หลัก |
| **5** | **PSK treatment** | **เคลือบ** |
| 6 | ETL | หลัก |
| **7** | **ETL treatment** | **เคลือบ** |
| 8 | Cathode | หลัก |

In [ ]:
MAIN_IDS    = {1, 2, 4, 6, 8}
COATING_IDS = {3, 5, 7}

_L = lay.dropna(subset=["material"]).copy()
_L["layer_kind"] = _L["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")

print("layer_id x layer_type (นับ device) + ชนิดที่จัดให้:")
_chk = (lay.groupby(["layer_id", "layer_type"]).device_id.nunique().rename("n_dev").reset_index())
_chk["kind"] = _chk["layer_id"].map(lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
display(_chk)

print("จำนวนชั้น (นับเฉพาะ material ไม่ว่าง) ต่อ device:")
_cnt = _L.groupby("device_id").agg(
    n_total=("layer_id", "nunique"),
    n_main=("layer_kind", lambda s: (s == "main").sum()),
    n_coating=("layer_kind", lambda s: (s == "coating").sum()))
display(_cnt["n_coating"].value_counts().sort_index().rename("n_devices").to_frame())

## 2. คอลัมน์ใหม่: เคลือบ / ไม่เคลือบ (ระดับ device)

ธงว่าแต่ละ device มีชั้นเคลือบตรงตำแหน่งไหนบ้าง + สรุปรวม

In [ ]:
dev = tp["devices"][["device_id"]].copy()

# ธงแยกแต่ละตำแหน่งเคลือบ
COAT_MAP = {3: "coat_htl", 5: "coat_psk", 7: "coat_etl"}
for lid, col in COAT_MAP.items():
    hit = _L.loc[_L["layer_id"] == lid, "device_id"].unique()
    dev[col] = dev["device_id"].isin(hit)

# สรุปรวม
dev["is_coated"] = dev[list(COAT_MAP.values())].any(axis=1)
dev["n_coating_layers"] = dev[list(COAT_MAP.values())].sum(axis=1)

print("device เคลือบ / ไม่เคลือบ:")
display(dev["is_coated"].value_counts().rename("n_devices").to_frame())
print("แยกตามตำแหน่ง:")
display(dev[list(COAT_MAP.values())].sum().rename("n_devices").to_frame())
display(dev.head())

## 3. layer 2, 4 ปิดด้วย Annealing จริงไหม

เช็คจาก `process` (`source_sheet == "Process"`) ว่า layer 2 (HTL) และ 4 (PSK) มีขั้น Annealing

In [ ]:
_P = pr[pr["source_sheet"] == "Process"]

for lid, name in [(2, "HTL"), (4, "PSK")]:
    g = _P[_P["layer_id"] == lid]
    ann = g[g["process_type"].str.contains("nneal", case=False, na=False)]
    print(f"layer {lid} ({name}): {ann.device_id.nunique()}/{g.device_id.nunique()} devices มี Annealing")

# ธงระดับ device
for lid, col in [(2, "anneal_htl"), (4, "anneal_psk")]:
    hit = _P[(_P["layer_id"] == lid) & _P["process_type"].str.contains("nneal", case=False, na=False)]["device_id"].unique()
    dev[col] = dev["device_id"].isin(hit)

display(dev[["anneal_htl", "anneal_psk"]].sum().rename("n_devices").to_frame())

## 4. แตก `process.parameter` + `unit` เป็นคอลัมน์แยก

`parameter` กับ `unit` เป็นค่าหลายตัวคั่น comma จับคู่ตามตำแหน่ง

```
parameter = "155, 600, 600"   unit = "W, mtorr, sec"
    -> W=155   mtorr=600   sec=600
```

ชื่อคอลัมน์ = `process_type` + `unit` เพราะหน่วยเดียวกันคนละความหมายตามขั้นตอน
(`C` ใน Annealing กับใน Dry คนละเรื่อง)

**⚠️ 9 แถว parameter พิมพ์ผิด** (ตัวเลขติดกัน / comma หาย) — 8 แถวแก้ใน §4.0 ด้วยค่ายืนยันจาก sibling sample ใน Raw Excel, เหลือ 1 แถวกำกวมติดธง `param_note` ไว้ ไม่เดา

### 4.0 แก้ parameter พิมพ์ผิดก่อนแตก

| ดิบ | แก้เป็น | หลักฐาน |
|---|---|---|
| `155600600` | `155, 600, 600` | S3 ไฟล์เดียวกันเขียน `155,600, 600` |
| `1556003600` | `155, 600, 3600` | รูปแบบเดียวกัน plasma นานขึ้น |
| `27010800` | `270, 10800` | S1 เขียน `270, 10800` |
| `603600` | `60, 3600` | S1 เขียน `60, 3600` |
| `80, 0.03 500` | `80, 0.03, 500` | comma หายหลัง 0.03 |

**ไม่แก้**: `100108000` (admi S3, L4 Annealing) — S1=`100, 1800` S4=`100, 10800` ขัดกัน รอเปิด paper ไม่เดา

In [ ]:
# ---- §4.0 แก้ parameter พิมพ์ผิด (ยืนยันจาก sibling sample ใน Raw Excel ไฟล์เดียวกัน) ----
PARAM_FIX = {
    # (device_id, layer_id, ค่าดิบ): ค่าที่ถูก
    ("10.1002admi.202500231_Sample1", 1, "155600600"):     "155, 600, 600",   # S3 เขียน `155,600, 600` ชัด
    ("10.1002admi.202500231_Sample4", 1, "1556003600"):    "155, 600, 3600",  # รูปแบบเดียวกัน plasma 3600 s
    ("10.1002admi.202500231_Sample3", 2, "27010800"):      "270, 10800",      # S1 เขียน `270, 10800`
    ("10.1002admi.202500231_Sample4", 2, "27010800"):      "270, 10800",
    ("10.1002admi.202500231_Sample3", 4, "603600"):        "60, 3600",        # S1 เขียน `60, 3600`
    ("10.1002admi.202500231_Sample4", 4, "603600"):        "60, 3600",
    ("10_1021_acsami_4c14300_Sample1", 2, "80, 0.03 500"): "80, 0.03, 500",   # comma หายหลัง 0.03
    ("10_1021_acsami_4c14300_Sample2", 2, "80, 0.03 500"): "80, 0.03, 500",
}
# ไม่แก้ (กำกวม): 10.1002admi.202500231_Sample3 L4 Annealing `100108000` (C, sec)
#   อ่านตรงตัว = 100, 108000 แต่ S1=`100, 1800` S4=`100, 10800` ขัดกัน -- รอเปิด paper ยืนยัน

pr["param_fixed"] = False
_lidn = pd.to_numeric(pr["layer_id"], errors="coerce")
for (d, l, bad), good in PARAM_FIX.items():
    m = (pr["device_id"] == d) & (_lidn == l) & (pr["parameter"].astype(str).str.strip() == bad)
    assert m.sum() == 1, f"PARAM_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} L{l} {bad!r}"
    pr.loc[m, "parameter"] = good
    pr.loc[m, "param_fixed"] = True

print(f"แก้แล้ว {int(pr['param_fixed'].sum())}/8 แถว | เหลือกำกวม 1 แถว (`100108000`) ปล่อยติดธงใน §4")

In [ ]:
UNIT_FIX = {"µL": "uL", "min each": "min"}          # รวมหน่วยพ้อง

def split_param(param, unit):
    """คืน dict {unit: value} + note ถ้าจำนวนไม่ตรง"""
    if pd.isna(param):
        return {}, ""
    ps = [t.strip() for t in str(param).split(",") if t.strip()]
    us = [UNIT_FIX.get(t.strip(), t.strip()) for t in str(unit).split(",") if t.strip()] if pd.notna(unit) else []
    if not us:
        return {}, ""
    if len(ps) != len(us):
        return {}, f"param {len(ps)} ค่า != unit {len(us)} ค่า: {param!r} | {unit!r}"
    out = {}
    for p, u in zip(ps, us):
        out[u] = pd.to_numeric(p, errors="coerce")
    return out, ""

parsed, notes = [], []
for _, r in pr.iterrows():
    d, note = split_param(r["parameter"], r["unit"])
    parsed.append(d)
    notes.append(note)

_pv = pd.json_normalize(parsed)                                  # แต่ละ unit เป็นคอลัมน์
_pv.columns = [f"{pt}__{u}" for pt, u in
               zip(pr["process_type"].fillna("NA"), [""] * len(_pv))] if False else _pv.columns
# ตั้งชื่อ process_type + unit ต่อแถว: ทำผ่านการ melt เพื่อความชัด
long = []
for i, (d, note) in enumerate(zip(parsed, notes)):
    ptype = pr.iloc[i]["process_type"]
    for u, v in d.items():
        long.append({"row": i, "col": f"{ptype}__{u}", "value": v})
proc_split = pd.DataFrame(long)

pr_out = pr.copy()
pr_out["param_note"] = notes
wide = proc_split.pivot_table(index="row", columns="col", values="value", aggfunc="first")
pr_out = pr_out.join(wide)

n_bad = int((pr_out["param_note"] != "").sum())
n_cols = wide.shape[1]
print(f"แตกเป็น {n_cols} คอลัมน์ | {n_bad} แถวติดธง (param/unit ไม่ตรง)")

print("\nแถวที่ติดธง (ตัวเลขติดกัน ต้องแก้ที่ต้นฉบับ):")
display(pr_out.loc[pr_out["param_note"] != "", ["device_id", "process_type", "parameter", "unit", "param_note"]]
        .drop_duplicates(subset=["parameter", "unit"]))

### 4.1 ตัวอย่างคอลัมน์ที่แตกได้ — process knob ที่เอาไปใช้ต่อได้

In [ ]:
_knob_cols = [c for c in pr_out.columns if "__" in c]
print("คอลัมน์ทั้งหมด:")
for c in sorted(_knob_cols):
    n = pr_out[c].notna().sum()
    if n:
        print(f"  {c:28s} {n:3d} แถว  ({pr_out[c].min():g} - {pr_out[c].max():g})")

print("\nตัวอย่าง Spin coating + Annealing:")
_show = [c for c in _knob_cols if c.startswith(("Spin coating", "Annealing"))]
display(pr_out.loc[pr_out[_show].notna().any(axis=1),
                   ["device_id", "layer_id", "process_type", "parameter", "unit"] + _show].head(10))

## 5. สรุป device-level ที่ได้เพิ่ม

รวมธง layer เข้า `dev` — เอาไป join กับ `ml` ใน `after_col_prepare` ได้

In [ ]:
cols = ["device_id", "is_coated", "n_coating_layers",
        "coat_htl", "coat_psk", "coat_etl", "anneal_htl", "anneal_psk"]
device_layer_flags = dev[cols]
print("device_layer_flags:", device_layer_flags.shape)
display(device_layer_flags.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

## 6. normalize เวลาเป็นวินาที + หมายเหตุ anneal temp

หน่วยเวลาปนกัน `sec` / `min` / `hr` รวมเป็น `sec` เดียว (`hr x3600`, `min x60`)

**anneal temp สูง (270-450C) ไม่ใช่บั๊ก** — เป็นการ anneal ชั้น NiOx/HTL (ปกติ 270-450C)
คนละตัวกับ anneal ชั้น PSK (100-150C) **ต้องแยกด้วย `layer_id` ตอนใช้เป็น feature** ห้ามรวม

In [ ]:
SEC_PER = {"sec": 1, "min": 60, "hr": 3600}

# รวมเวลาต่อ process_type: <ptype>__time_s = sec + min*60 + hr*3600
_time_units = ["sec", "min", "hr"]
_ptypes = sorted({c.split("__")[0] for c in pr_out.columns if "__" in c})
for pt in _ptypes:
    parts = [(f"{pt}__{u}", SEC_PER[u]) for u in _time_units if f"{pt}__{u}" in pr_out.columns]
    if len(parts) > 1 or (parts and parts[0][0].split("__")[1] != "sec"):
        s = pd.Series(0.0, index=pr_out.index)
        any_val = pd.Series(False, index=pr_out.index)
        for col, mult in parts:
            s = s + pr_out[col].fillna(0) * mult
            any_val = any_val | pr_out[col].notna()
        pr_out[f"{pt}__time_s"] = s.where(any_val)

# ตรวจ anneal แยกตาม layer
_ann = pr_out[pr_out["process_type"] == "Annealing"].dropna(subset=["Annealing__C"])
print("anneal temp แยกตาม layer_id (เห็นว่าชั้นล่าง NiOx/HTL ร้อนกว่า PSK):")
display(_ann.groupby("layer_id")["Annealing__C"].agg(["count", "min", "median", "max"]))

### 6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา

- `__ml` กับ `__uL` เป็น knob เดียวกัน → `AntiSolvent__vol_uL` (0.25 ml = 250 uL)
- เวลาหยด: **ค่าลบ = วินาทีก่อนจบ spin, ค่าบวก = วินาทีหลังเริ่ม** คนละจุดอ้างอิง
  เก็บ `AntiSolvent__timing_ref` กำกับ — ใช้เป็น feature ต้องแยกกลุ่ม หรือแปลงด้วย spin duration ก่อน ห้ามรวมสเกลตรง ๆ

In [ ]:
# ---- §6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา ----
# ปริมาตร: __ml (4 dev) กับ __uL (31 dev) เป็น knob เดียวกัน -> uL เดียว
vol = pd.Series(np.nan, index=pr_out.index)
if "AntiSolvent__uL" in pr_out.columns:
    vol = pr_out["AntiSolvent__uL"].copy()
if "AntiSolvent__ml" in pr_out.columns:
    vol = vol.fillna(pr_out["AntiSolvent__ml"] * 1000)
pr_out["AntiSolvent__vol_uL"] = vol

# เวลา: ค่าลบ = วินาที "ก่อนจบ" spin, ค่าบวก = วินาที "หลังเริ่ม" -- คนละจุดอ้างอิง ห้ามรวมสเกลตรง ๆ
t = pr_out["AntiSolvent__sec"] if "AntiSolvent__sec" in pr_out.columns else pd.Series(np.nan, index=pr_out.index)
pr_out["AntiSolvent__timing_s"] = t
pr_out["AntiSolvent__timing_ref"] = pd.Series(
    np.where(t < 0, "before_end", "after_start"), index=pr_out.index).where(t.notna())

_m = pr_out["AntiSolvent__vol_uL"].notna()
print(f"AntiSolvent__vol_uL: {pr_out.loc[_m, 'device_id'].nunique()} devices "
      f"(uL {int(pr_out['AntiSolvent__uL'].notna().sum())} แถว + ml {int(pr_out['AntiSolvent__ml'].notna().sum())} แถว)")
print("timing_ref:", pr_out["AntiSolvent__timing_ref"].value_counts().to_dict())

## 7. ฝั่ง Y: เลือก target จาก `properties`

**2 ชั้น แยกกัน**

**Y ประสิทธิภาพ (target หลัก)** — เทรน Voc/Jsc/FF แยกแล้วคูณเป็น PCE
เพราะ `PCE = Voc x Jsc x FF / 100` (พิสูจน์แล้ว corr 0.996) เทรน PCE ตรงไม่ได้ข้อมูลใหม่ แยกได้กลไก

**mediator เชิงกายภาพ** — ใช้ **VTFL** เป็นตัวหลัก (29 dev เยอะกว่า grain_size 13 และ trap_density 23)
VTFL วัดข้อบกพร่องเหมือน trap density แต่เป็นค่าดิบ coverage ดีกว่า

In [ ]:
prop = tp["properties"].dropna(subset=["value_parsed"]).copy()
prop["paper"] = prop["device_id"].str.rsplit("_Sample", n=1).str[0]

TARGETS  = ["PCE", "Voc", "Jsc", "FF"]
MEDIATOR = ["VTFL", "Trap density", "Rs", "Rrec", "Contact angle", "Grain size"]

cov = (prop[prop["base_property"].isin(TARGETS + MEDIATOR)]
       .groupby("base_property")
       .agg(rows=("value_parsed", "size"),
            devices=("device_id", "nunique"),
            papers=("paper", "nunique"),
            lo=("value_parsed", "min"), hi=("value_parsed", "max")))
cov["บทบาท"] = cov.index.map(lambda b: "target" if b in TARGETS else "mediator")
cov["ใช้"] = cov["devices"].ge(15) & cov["papers"].ge(5)
display(cov.sort_values(["บทบาท", "devices"], ascending=[True, False]))

# ---- JV: เลือกชุดเดียวกัน (variant-consistent) ----
# บั๊กเดิม: aggfunc="max" ต่อ property อิสระ -> บาง paper ได้ Champion PCE ปน Average Voc/Jsc/FF
# (เช่น d3lf00258f รายงาน component เป็น Average อย่างเดียว) ทำให้ pce != voc*jsc*ff/100
_jv = prop[prop["base_property"].isin(TARGETS)].copy()
_p = _jv["property"].fillna("")
_jv["variant"] = np.where(_p.str.contains("hampion"), "champion",
                 np.where(_p.str.contains("verage"), "average", "plain"))

piv = {v: (_jv[_jv["variant"] == v]
           .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
           .reindex(columns=TARGETS))
       for v in ["champion", "plain", "average"]}

rows = []
for dev in sorted(_jv["device_id"].unique()):
    pick, r = None, None
    for v in ["champion", "plain", "average"]:          # ชุดแรกที่มี voc+jsc+ff ครบ
        if dev in piv[v].index and piv[v].loc[dev, ["Voc", "Jsc", "FF"]].notna().all():
            pick, r = v, piv[v].loc[dev]
            break
    if pick is None:                                     # ไม่มีชุดครบใน variant เดียว -> รวมทุกแถว ติดธง
        pick = "mixed"
        r = (_jv[_jv["device_id"] == dev]
             .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
             .reindex(columns=TARGETS).loc[dev])
    rows.append({"device_id": dev, "jv_variant": pick,
                 **{c.lower(): r[c] for c in TARGETS}})
jv = pd.DataFrame(rows).set_index("device_id")

print("jv_variant:", jv["jv_variant"].value_counts().to_dict())
_have = jv.dropna(subset=["voc", "jsc", "ff", "pce"])
_err = (_have["voc"] * _have["jsc"] * _have["ff"] / 100 - _have["pce"]).abs()
# เกณฑ์ 1.0: ค่ารายงานปัด 2 ตำแหน่ง ทำ product คลาดได้ ~0.5 (เช่น ccschem champion) -- ไม่ใช่ปน variant
print(f"เช็ค pce = voc*jsc*ff/100: n={len(_have)} max_err={_err.max():.3f} | เกิน 1.0: {int((_err > 1.0).sum())} เครื่อง (ควรเป็น 0)")

# mediator ที่ผ่านเกณฑ์
med = (prop[prop["base_property"].isin(["VTFL", "Trap density", "Rs", "Rrec"])]
       .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="median"))
med.columns = [c.lower().replace(" ", "_") for c in med.columns]

Y = jv.join(med)
print(f"\nตาราง Y: {Y.shape[0]} devices x {Y.shape[1]} คอลัมน์")
display(Y.describe(include=[np.number]).T.round(3))

## 8. Stability: คิด index ก่อนใช้เป็น Y

retention ดิบใช้ตรงไม่ได้ — วัดคนละสภาพ (25-85C, 1-5000 ชม.)

ทำ 2 อย่าง
1. `degradation_rate` = `(100 - retention) / time_hr` [%/ชม.] — normalize เวลา
2. `temp_bin` — จับกลุ่มอุณหภูมิ ใช้เทียบเฉพาะในกลุ่มเดียวกัน

**ข้อจำกัด**: การเสื่อมไม่เชิงเส้น (burn-in เร็วแล้วช้า) rate เทียบข้าม timescale ตรง ๆ ไม่ได้
ใช้ retention @ 25C bin เป็น Y เสถียร (14 dev) เป็นทางที่สะอาดสุด แต่ยัง exploratory

In [ ]:
stab = tp["stability"].copy()
stab["ret"] = pd.to_numeric(stab["retention_pct_parsed"], errors="coerce")
stab["t_hr"] = pd.to_numeric(stab["time_hr"], errors="coerce")
stab["Tp"] = pd.to_numeric(stab["temperature_c_parsed"], errors="coerce")
stab = stab.dropna(subset=["ret"])

stab["degradation_rate"] = (100 - stab["ret"]) / stab["t_hr"]         # %/hr
stab["temp_bin"] = pd.cut(stab["Tp"], [0, 30, 60, 100],
                          labels=["~25C", "45-50C", "65-85C"])

print("index รายกลุ่มอุณหภูมิ:")
display(stab.groupby("temp_bin", observed=True).agg(
    devices=("device_id", "nunique"),
    ret_median=("ret", "median"),
    rate_median=("degradation_rate", "median")).round(3))
print(f"ไม่มี temp: {int(stab['Tp'].isna().sum())} แถว (ใช้ได้เฉพาะ degradation_rate)")

# Y เสถียร: retention ที่ ~25C bin (สภาพใกล้กันสุด)
stab_Y = (stab[stab["temp_bin"] == "~25C"]
          .groupby("device_id").agg(retention_25c=("ret", "max"),
                                    degradation_rate=("degradation_rate", "min")))
print(f"\nY เสถียร (retention @ 25C): {len(stab_Y)} devices")
display(stab_Y.head())

## 8.5 PSK composition (a_Cs / a_FA / x_Br)

`column_prepare` §J คำนวณครบ 44 devices แต่**ไม่เคย export ลงดิสก์** — port ตรรกะเดิมทุกบรรทัด (J1-J5, TOL=0.03) มาคำนวณใหม่จากตาราง prepared ที่นี่

สูตรนามเป็นหลัก → precursor เติมช่องว่าง + cross-check | export เป็น `psk_composition.csv` ใน §10

In [ ]:
# ---- §8.5 psk_composition -- port ตรรกะเดิมจาก column_prepare §J (J1-J5) ----
# column_prepare คำนวณแล้วแต่ไม่เคยเขียนดิสก์ -- คำนวณใหม่จากตาราง prepared ที่นี่แล้ว export ใน §10

# J1: อ่านสัดส่วนจากสูตรในชื่อชั้น
CATIONS = ["FA", "MA", "Cs", "Rb", "Pb", "Sn", "Br", "Cl", "I", "K"]   # ยาว->สั้น กัน FA ถูกอ่านเป็น F+A
TOKEN = re.compile("(" + "|".join(CATIONS) + r")(\d*\.?\d+)")

def _site_from_str(part):
    out = {}
    for inner, scale in re.findall(r"\(([^)]*)\)\s*(\d*\.?\d+)?", part):
        k = float(scale) if scale else 1.0
        for ion, frac in TOKEN.findall(inner):
            out[ion] = out.get(ion, 0) + float(frac) * k
    rest = re.sub(r"\([^)]*\)\s*\d*\.?\d*", "", part)
    for ion, frac in TOKEN.findall(rest):
        out[ion] = out.get(ion, 0) + float(frac)
    return out

def composition_from_formula(f):
    s = str(f).strip().strip("[]").strip()
    m = re.search(r"(Pb|Sn)", s)
    if not m:
        return None
    A, X = _site_from_str(s[:m.start()]), _site_from_str(s[m.end():])
    sA, sX = sum(A.values()), sum(X.values())
    if not sA or not sX:
        return None
    return ({k: v / sA for k, v in A.items()}, {k: v / sX for k, v in X.items()}, m.group(1))

# J2: molar mass + ไอออนที่สารตั้งต้นให้
MW = {"CsI": 259.81, "CsBr": 212.81, "FAI": 171.97, "MAI": 158.97, "MABr": 111.97,
      "MACl": 67.52, "PbI2": 461.01, "PbBr2": 367.01, "PbCl2": 278.11}
ION = {"CsI":  {"A": {"Cs": 1}, "X": {"I": 1}},   "CsBr": {"A": {"Cs": 1}, "X": {"Br": 1}},
       "FAI":  {"A": {"FA": 1}, "X": {"I": 1}},   "MAI":  {"A": {"MA": 1}, "X": {"I": 1}},
       "MABr": {"A": {"MA": 1}, "X": {"Br": 1}},  "MACl": {"A": {"MA": 1}, "X": {"Cl": 1}},
       "PbI2": {"B": {"Pb": 1}, "X": {"I": 2}},   "PbBr2":{"B": {"Pb": 1}, "X": {"Br": 2}},
       "PbCl2":{"B": {"Pb": 1}, "X": {"Cl": 2}}}
MOLAR = {"m": 1.0, "mol": 1.0, "m total": 1.0, "mmol": 0.001}
MASS  = {"mg": 0.001, "g": 1.0, "g / 2ml": 1.0}
VOLATILE = {"MACl"}

def canon_precursor(s):
    k = re.sub(r"[^a-z0-9]", "", str(s).lower())
    for n in MW:
        if re.sub(r"[^a-z0-9]", "", n.lower()) == k:
            return n
    return None

def unit_family(u):
    u = str(u).strip().lower()
    return "molar" if u in MOLAR else ("mass" if u in MASS else None)

# J3: join ชั้น PSK กับสารตั้งต้น
psk = lay.loc[lay["layer_type"] == "PSK",
              ["device_id", "layer_id", "material"]].rename(columns={"material": "layer_material"})
jm = psk.merge(tp["materials"], on=["device_id", "layer_id"], how="left")
jm["prec"] = jm["material"].map(canon_precursor)
jm = jm.dropna(subset=["prec"]).copy()
jm = jm[~jm["prec"].isin(VOLATILE)]
jm["ufam"] = jm["unit"].map(unit_family)
jm["amt"]  = pd.to_numeric(jm["amount"], errors="coerce")

# J4: คำนวณจากสารตั้งต้นทีละ device
prec_comp, prec_status = {}, {}
for dev, g in jm.groupby("device_id"):
    fams = set(g["ufam"].dropna())
    bad  = g[g["ufam"].isna() | g["amt"].isna()]
    if len(bad):
        prec_status[dev] = "ตกหล่น: " + ", ".join(sorted(set(bad["prec"])))
    elif len(fams) != 1:
        prec_status[dev] = f"หน่วยปนตระกูล {sorted(fams)}"
    elif not any(ION[p].get("B") for p in g["prec"]):
        prec_status[dev] = "ไม่มี B-site (สูตรไม่ครบ)"
    else:
        fam, A, X = fams.pop(), {}, {}
        conv = MOLAR if fam == "molar" else MASS
        for _, r in g.iterrows():
            mol = r["amt"] * conv[str(r["unit"]).strip().lower()]
            if fam == "mass":
                mol /= MW[r["prec"]]
            spec = ION[r["prec"]]
            for i, mlt in spec.get("A", {}).items(): A[i] = A.get(i, 0) + mol * mlt
            for i, mlt in spec.get("X", {}).items(): X[i] = X.get(i, 0) + mol * mlt
        sA, sX = sum(A.values()), sum(X.values())
        prec_comp[dev] = ({k: v/sA for k, v in A.items()}, {k: v/sX for k, v in X.items()}, "Pb")
        prec_status[dev] = f"ok ({fam})"

# J5: รวม 2 แหล่ง สูตรเป็นหลัก precursor เติมช่องว่าง + cross-check
TOL = 0.03
rows = []
for _, r in psk.iterrows():
    dev = r["device_id"]
    f_res, p_res = composition_from_formula(r["layer_material"]), prec_comp.get(dev)
    row = {"device_id": dev, "layer_material": r["layer_material"],
           "precursor_status": prec_status.get(dev, "ไม่มีแถว precursor")}
    if f_res:
        A, X, b = f_res
        row["comp_source"] = "formula"
        if p_res:
            pA, pX, _ = p_res
            diff = [k for k in set(A) | set(X) | set(pA) | set(pX)
                    if abs({**A, **X}.get(k, 0) - {**pA, **pX}.get(k, 0)) > TOL]
            row["cross_check"] = "ตรง" if not diff else "ต่าง: " + ", ".join(sorted(diff))
    elif p_res:
        A, X, b = p_res
        row["comp_source"] = "precursor"
        row["cross_check"] = "-- (สูตรแตกไม่ได้)"
    else:
        row["comp_source"] = "ไม่มีข้อมูล"
        rows.append(row)
        continue
    row["b_site"] = b
    row.update({f"a_{k}": round(v, 3) for k, v in A.items()})
    row.update({f"x_{k}": round(v, 3) for k, v in X.items()})
    rows.append(row)

psk_composition = pd.DataFrame(rows)
front = ["device_id", "comp_source", "cross_check", "b_site"]
psk_composition = psk_composition[[c for c in front if c in psk_composition.columns]
                                  + [c for c in psk_composition.columns if c not in front]]

n_ok = int((psk_composition["comp_source"] != "ไม่มีข้อมูล").sum())
print(f"องค์ประกอบ {n_ok}/{psk_composition['device_id'].nunique()} devices | "
      f"source: {psk_composition['comp_source'].value_counts().to_dict()}")

# device-level สำหรับ merge เข้า target_table (ตามแผน §9: a_Cs / a_FA / x_Br)
# ไอออนที่ไม่อยู่ในสูตร = สัดส่วน 0 จริง ไม่ใช่ missing (เช่น pure iodide -> x_Br = 0)
# fill 0 เฉพาะ device ที่คำนวณองค์ประกอบได้ -- ที่คำนวณไม่ได้คง NaN
_cd = psk_composition.drop_duplicates("device_id").set_index("device_id")
comp_dev = _cd.reindex(columns=["a_Cs", "a_FA", "x_Br"])
_ok = _cd["comp_source"] != "ไม่มีข้อมูล"
comp_dev.loc[_ok] = comp_dev.loc[_ok].fillna(0.0)
print("coverage (หลัง fill 0 ให้ไอออนที่ไม่อยู่ในสูตร):", comp_dev.notna().sum().to_dict())

## 9. สรุปสิ่งที่ควรใช้

| ฝั่ง | ใช้ | จำนวน | หมายเหตุ |
|---|---|---|---|
| **Y ประสิทธิภาพ** | `voc` `jsc` `ff` แยก คูณเป็น pce | 43 dev / 15 paper | variant-consistent (`jv_variant` กำกับ champion/plain/average) |
| **mediator** | `vtfl` | ~28 dev / 11 paper | แทน grain_size (13) |
| **Y เสถียร** | `retention_25c` | 14 dev | exploratory เท่านั้น |
| **X knob** | `psk_spin_rpm` `psk_anneal_C` `psk_anneal_time_s` `antisolvent_vol_uL` `antisolvent_timing_s` `htl_anneal_C` | 30-43 dev | §4-6.1 merge เข้า target_table แล้ว |
| **X ควบคุม** | `a_Cs` `a_FA` `x_Br` | ~44 dev | §8.5 (port จาก column_prepare §J) กัน confound สูตรเคมี |

**ตัดทิ้ง**: PCE ตรง (= ผลคูณ), grain_size แกนหลัก, t80, has_antisolvent/etl (paper เดียว), *_family variance=0

**ระวัง**: `antisolvent_timing_s` มี 2 จุดอ้างอิง (ดู `antisolvent_timing_ref`) ห้ามใช้เป็น feature เดียวตรง ๆ

**โมเดล**: RandomForest + SHAP, LeaveOneGroupOut(paper) 15 fold, รายงาน mean±SD + bootstrap, ผล = hypothesis

In [ ]:
# ---- X: knob ระดับ device (PSK layer 4 + HTL layer 2) ----
_lidn = pd.to_numeric(pr_out["layer_id"], errors="coerce")
_pskp = pr_out[_lidn == 4]
_htlp = pr_out[_lidn == 2]

def _knob(df, ptype, col, agg):
    if col not in df.columns:
        return pd.Series(dtype=float)
    g = df[df["process_type"] == ptype].groupby("device_id")[col]
    return getattr(g, agg)()

X_dev = pd.DataFrame(index=device_layer_flags.set_index("device_id").index)
X_dev["psk_spin_rpm"]         = _knob(_pskp, "Spin coating", "Spin coating__RPM", "max")   # step สุดท้าย = RPM สูงสุด
X_dev["psk_anneal_C"]         = _knob(_pskp, "Annealing", "Annealing__C", "max")
X_dev["psk_anneal_time_s"]    = _knob(_pskp, "Annealing", "Annealing__time_s", "sum")
X_dev["antisolvent_vol_uL"]   = _knob(_pskp, "AntiSolvent", "AntiSolvent__vol_uL", "max")
X_dev["antisolvent_timing_s"] = _knob(_pskp, "AntiSolvent", "AntiSolvent__timing_s", "first")
X_dev["antisolvent_timing_ref"] = (_pskp.dropna(subset=["AntiSolvent__timing_s"])
                                   .groupby("device_id")["AntiSolvent__timing_ref"].first())
X_dev["htl_anneal_C"]         = _knob(_htlp, "Annealing", "Annealing__C", "max")

# target_table = flags + Y + stability + X knob + composition
target_table = (device_layer_flags.set_index("device_id")
                .join(Y).join(stab_Y).join(X_dev).join(comp_dev))

assert len(target_table) == 44 and target_table.index.is_unique, "target_table ต้อง 44 แถว unique"
print("target_table:", target_table.shape)
print("\ncoverage ต่อคอลัมน์ (devices):")
display(target_table.notna().sum().rename("n_devices").to_frame().T)
display(target_table.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

## 10. Export `prepared_v2` — ชุดเต็มทุกไฟล์

`prepared_v2/` = ทุกตารางจาก `prepared/` + สิ่งที่ recheck เพิ่ม

| ไฟล์ | เปลี่ยนจาก `prepared/` |
|---|---|
| `layers.csv` | + `layer_kind` (map ตรงจาก layer_id — **ไม่ merge** กันแถวพองเคส layer 7 สองแถว PCBM+BCP) |
| `process.csv` | + knob columns ที่แตก + `param_fixed` (§4.0) + `param_note` + `AntiSolvent__vol_uL/timing_s/timing_ref` (§6.1) + `*__time_s` (§6) |
| `devices.csv` | + layer flags (is_coated, coat_*, anneal_*) |
| `materials/properties/stability/vendors.csv` | เหมือนเดิม |
| `target_table.csv` | ใหม่ — Y (variant-consistent) + mediator + stability + **X knob + composition** |
| `device_layer_flags.csv` | ใหม่ |
| `psk_composition.csv` | ใหม่ — §8.5 (เดิมคำนวณใน column_prepare แต่ไม่เคย export) |

In [ ]:
V2 = ROOT / "processed" / "prepared_v2"
V2.mkdir(parents=True, exist_ok=True)

# 1) layers + layer_kind -- map ตรงจาก layer_id ห้าม merge
#    (solr_202400534 มี layer 7 สองแถว PCBM+BCP -- merge บน (device_id, layer_id) จะคูณแถวเป็น 2x2)
_lay_out = lay.copy()
_kind = _lay_out["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
_lay_out["layer_kind"] = _kind.where(_lay_out["material"].notna())
assert len(_lay_out) == len(lay), "layers แถวพอง"
_lay_out.to_csv(V2 / "layers.csv", index=False)

# 2) process = แตก parameter + PARAM_FIX + antisolvent + time_s
assert int((pr_out["param_note"] != "").sum()) == 1, "ควรเหลือแถวติดธงเดียว (100108000)"
pr_out.to_csv(V2 / "process.csv", index=False)

# 3) devices + layer/anneal flags
_dev_out = tp["devices"].merge(device_layer_flags, on="device_id", how="left")
_dev_out.to_csv(V2 / "devices.csv", index=False)

# 4) ตารางที่เหลือ copy ตรง
for name in ["materials", "properties", "stability", "vendors"]:
    tp[name].to_csv(V2 / f"{name}.csv", index=False)

# 5) ตารางที่ recheck สร้าง
target_table.to_csv(V2 / "target_table.csv")
device_layer_flags.to_csv(V2 / "device_layer_flags.csv", index=False)
psk_composition.to_csv(V2 / "psk_composition.csv", index=False)

print("wrote prepared_v2:")
for p in sorted(V2.glob("*.csv")):
    d = pd.read_csv(p)
    print(f"  {p.name:24s} {d.shape}")

# verify จากไฟล์บนดิสก์
_chk_lay = pd.read_csv(V2 / "layers.csv")
assert len(_chk_lay) == len(lay), "layers บนดิสก์แถวไม่ตรง"
_chk_tt = pd.read_csv(V2 / "target_table.csv")
assert len(_chk_tt) == 44 and _chk_tt["device_id"].is_unique, "target_table บนดิสก์ไม่ใช่ 44 unique"
_hv = _chk_tt.dropna(subset=["voc", "jsc", "ff", "pce"])
_e = (_hv["voc"] * _hv["jsc"] * _hv["ff"] / 100 - _hv["pce"]).abs()
assert int((_e > 1.0).sum()) == 0, "ยังมี pce ปน variant"   # 1.0 เผื่อ rounding ค่ารายงาน (บั๊กเดิม err 1.64)
print(f"verify ผ่าน: layers {len(_chk_lay)} แถวไม่พอง | target_table 44 unique | pce consistency max_err={_e.max():.3f}")